# 09 Build a Basic Transformer Block

## Problem

前面学过的 attention、RMSNorm、FFN 具体是怎样拼成一个 decoder-only Transformer block 的？为什么顺序通常是 norm -> attention -> residual -> norm -> FFN -> residual？

## Dependencies

建议先完成 `04` 到 `08`。至少要知道 self-attention、RoPE、RMSNorm、residual、FFN 各自是干什么的。


## Goals

- 理解一个基础 Transformer block 的数据流顺序
- 理解 pre-norm decoder block 的结构
- 能追踪每一步的 shape 变化
- 能把一个最小 block 的前向过程完整讲出来

## Scope and Approach

这一节的重点不是把几个零件硬拼在一起，而是回答：为什么是这个顺序？如果把顺序打乱会怎样？我们会先把 block 的结构图讲清，再用一个最小实现把前向过程完整跑一遍。


## 组装 Transformer block 时最容易糊的地方

前面每个零件单独看都还清楚，一到拼装阶段，很多人就开始乱：

- attention 前要不要先 norm？
- residual 是加在子层前还是后？
- FFN 和 attention 谁先谁后？

现代 decoder-only LLM 常见的基础节奏可以先记成：

1. 输入先做 norm
2. 进入 causal self-attention
3. attention 输出加回原输入，形成第一次 residual
4. 再做一次 norm
5. 进入 FFN
6. FFN 输出再加回去，形成第二次 residual

这就是一个最基础的 pre-norm block。


In [ ]:
import numpy as np

np.random.seed(11)
np.set_printoptions(precision=3, suppress=True)

def rms_norm(x, eps=1e-6):
    # x.shape = (..., hidden_dim)
    rms = np.sqrt((x ** 2).mean(axis=-1, keepdims=True) + eps)
    return x / rms

def softmax(logits):
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=-1, keepdims=True)

def causal_attention(x, W_q, W_k, W_v, W_o):
    # x.shape = (seq_len, hidden_dim)
    Q = x @ W_q
    K = x @ W_k
    V = x @ W_v

    scores = (Q @ K.T) / np.sqrt(Q.shape[-1])
    mask = np.triu(np.ones_like(scores), k=1) * -1e9
    weights = softmax(scores + mask)

    # attention 输出先是 weights @ V，再经过输出投影 W_o
    return (weights @ V) @ W_o

def swiglu_ffn(x, W_gate, W_value, W_down):
    sigmoid = lambda z: 1 / (1 + np.exp(-z))
    swish = lambda z: z * sigmoid(z)
    gate = swish(x @ W_gate)
    value = x @ W_value
    return (gate * value) @ W_down


## 先搭一个最小 block

这里我们用一个非常小的例子来把所有零件串起来：

- `seq_len = 4`
- `hidden_dim = 6`
- `ffn_dim = 12`

重点不是模型强不强，而是你能看清：

- 输入经过了哪些子层
- 每一步的 shape 有没有变
- residual 是在哪个时刻加回去的


In [ ]:
seq_len = 4
hidden_dim = 6
ffn_dim = 12

# 输入 shape = (seq_len, hidden_dim)
x = np.random.randn(seq_len, hidden_dim) * 0.2

# attention 所需参数
W_q = np.random.randn(hidden_dim, hidden_dim) * 0.2
W_k = np.random.randn(hidden_dim, hidden_dim) * 0.2
W_v = np.random.randn(hidden_dim, hidden_dim) * 0.2
W_o = np.random.randn(hidden_dim, hidden_dim) * 0.2

# FFN 所需参数
W_gate = np.random.randn(hidden_dim, ffn_dim) * 0.2
W_value = np.random.randn(hidden_dim, ffn_dim) * 0.2
W_down = np.random.randn(ffn_dim, hidden_dim) * 0.2

print('input.shape =', x.shape)

# 第 1 步：先做 norm，再进 attention
attn_input = rms_norm(x)
attn_output = causal_attention(attn_input, W_q, W_k, W_v, W_o)

# 第 2 步：把 attention 输出加回原输入，形成第一次 residual
x = x + attn_output
print('after attention residual =', x.shape)

# 第 3 步：再做一次 norm，再进 FFN
ffn_input = rms_norm(x)
ffn_output = swiglu_ffn(ffn_input, W_gate, W_value, W_down)

# 第 4 步：FFN 输出再加回去，形成第二次 residual
x = x + ffn_output
print('after ffn residual =', x.shape)
print('block output =')
print(x)


## 为什么顺序通常是这样

这一套顺序不是随便拍脑袋拼出来的。可以先抓住几点：

- 先 norm：让子层看到更稳定的输入分布
- 先 attention 后 FFN：先做跨位置交互，再做每个位置内部加工
- 每个子层后都 residual：保留原始信息通路，避免一层把表示彻底改坏

如果你把顺序完全打乱，就会失去这种“先交换信息、再内部加工、每一步都保留捷径”的结构节奏。


## 一个 block 和多个 block 的关系

理解一个 block 后，你就已经抓住了 Transformer 的重复单元。真正的大模型只是把这个单元堆很多层：

- 结构上每层类似
- 参数上每层通常独立
- 表示会一层层被重组和加工

所以 Transformer 的复杂性，很大一部分来自“重复堆叠 + 大规模训练”，而不是单个 block 本身就神秘得无法拆开。

## Common mistakes

- 把 block 记成一堆层名，而不去跟踪“数据是怎么流动的”。
- 忘记 causal mask，导致 block 不再符合自回归生成约束。
- 看到 FFN 和 attention 都有 residual，就以为次序可以随便换。先记住标准流程，再回头比较其他变体会更稳妥。
- 以为一个 block 的意义在于层名数量，而不是它如何把跨位置交互和单位置加工组合起来。

## Checkpoints

- 自己把 block 画成流程图，并标上每一步的 shape。
- 尝试把 `hidden_dim` 改成 8，看参数矩阵尺寸如何变化。
- 思考如果堆叠 12 个这样的 block，哪些模块会重复，哪些参数会各层独立。
- 用自己的话解释为什么 attention 和 FFN 要分成两个子层，而不是混在一起。

## Summary

当我们把这些零件拼起来，Transformer 就不再是神秘黑盒，而是一个结构非常清楚的重复模块：先 norm，再 attention，再 residual；再 norm，再 FFN，再 residual。真正的大模型，只是在这个基本节奏上不断堆深、放大、训练。

## Next Module

下一节进入推理侧的重要主题：KV cache。它解释了为什么自回归生成如果不缓存，会做很多重复劳动。
